In [ ]:
!pip -q install -U transformers accelerate datasets bitsandbytes scipy scikit-learn matplotlib

In [ ]:
import os
try:
    from google.colab import drive
    drive.mount('/content/drive')
    PROJECT_DIR = '/content/drive/MyDrive/LPP/data'   # <-- folder holding final_data.csv, ar_results_*, etc.
except Exception:
    PROJECT_DIR = os.path.join('..', 'data')   # local run from notebooks/
os.makedirs(PROJECT_DIR, exist_ok=True); os.chdir(PROJECT_DIR)
print('Working dir:', os.getcwd())
# !nvidia-smi -L

In [ ]:
# Hugging Face auth (needed for Llama/Mistral gated repos)
import os
os.environ.setdefault('HF_TOKEN', '')   # set HF_TOKEN in your environment; never commit tokens
!hf auth login --token $HF_TOKEN

In [ ]:
import os, numpy as np, pandas as pd
np.random.seed(42)

MODEL_MAP = {
    "Qwen-0.5B": "Qwen/Qwen2.5-0.5B-Instruct",
    "Qwen-1.5B": "Qwen/Qwen2.5-1.5B-Instruct",
    "Qwen-3B":   "Qwen/Qwen2.5-3B-Instruct",
    "Qwen-7B":   "Qwen/Qwen2.5-7B-Instruct",
    "Qwen-14B":  "Qwen/Qwen2.5-14B-Instruct",
    "Llama-3B":  "meta-llama/Llama-3.2-3B-Instruct",
    "Llama-8B":  "meta-llama/Meta-Llama-3-8B-Instruct",
    "Mistral-7B":"mistralai/Mistral-7B-Instruct-v0.2",
}
LPP_TABLE   = "final_data.csv"   # Model, IFEval, BBH, MMLU-PRO, Entropy, Participation Ratio, Effective Rank, SPC, AR
N_TRUTHFULQA = 200               # small subset
N_SYMB       = 120               # examples per symbolic task (Dyck / List-Functions)
K_SHOT       = 5                 # few-shot demos for Dyck
import torch
device   = "cuda" if torch.cuda.is_available() else "cpu"
USE_4BIT = True                  # 4-bit so 7B/14B fit on a free/L4 Colab GPU (set False on A100)

lpp = pd.read_csv(LPP_TABLE).rename(columns={
    "Participation Ratio":"PR", "Effective Rank":"ER", "Entropy":"EntropyFloor"})
lpp[["Model","EntropyFloor","PR","ER"]]

In [ ]:
template = pd.DataFrame({
    "Model": list(MODEL_MAP.keys()),
    "TruthfulQA_MC2": np.nan,   # higher = more truthful (less hallucination)
    "HaluEval_acc":   np.nan,   # higher = less hallucination
    "AdvGLUE_acc":    np.nan,   # higher = more adversarially resilient
})
if not os.path.exists("external_scores.csv"):
    template.to_csv("external_scores.csv", index=False)
    print("Wrote template external_scores.csv — fill it in, then rerun the next cell.")
else:
    print("external_scores.csv already exists; not overwriting.")
template

In [ ]:
# Load whatever published scores you have entered
ext_pub = pd.read_csv("external_scores.csv")
ext_pub = ext_pub.dropna(axis=1, how="all")          # drop columns still fully empty
print("Published external columns available:", [c for c in ext_pub.columns if c != "Model"])
ext_pub

In [ ]:
import torch, torch.nn.functional as F
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig

def load_model(hf_id, use_4bit=USE_4BIT):
    tok = AutoTokenizer.from_pretrained(hf_id, use_fast=True)
    if tok.pad_token is None: tok.pad_token = tok.eos_token
    tok.padding_side = "left"
    kw = dict(device_map="auto")
    if use_4bit and device == "cuda":
        kw["quantization_config"] = BitsAndBytesConfig(
            load_in_4bit=True, bnb_4bit_quant_type="nf4", bnb_4bit_use_double_quant=True,
            bnb_4bit_compute_dtype=torch.bfloat16 if torch.cuda.is_bf16_supported() else torch.float16)
    else:
        kw["torch_dtype"] = torch.float16
    model = AutoModelForCausalLM.from_pretrained(hf_id, **kw)
    model.eval()
    return tok, model

@torch.no_grad()
def seq_logprob(model, tok, context, continuation):
    """Length-normalized log P(continuation | context) — standard MC scoring."""
    ctx_ids = tok(context, return_tensors="pt").input_ids.to(model.device)
    full_ids = tok(context + " " + continuation, return_tensors="pt").input_ids.to(model.device)
    cont_ids = full_ids[:, ctx_ids.shape[1]:]
    if cont_ids.shape[1] == 0:
        return -1e9
    out = model(full_ids)
    logits = out.logits[:, ctx_ids.shape[1]-1:-1, :]
    logp = F.log_softmax(logits, dim=-1)
    tok_lp = logp.gather(-1, cont_ids.unsqueeze(-1)).squeeze(-1)
    return float(tok_lp.mean())     # length-normalized

In [ ]:
from datasets import load_dataset

TQA_PARQUET = ("hf://datasets/EleutherAI/truthful_qa_mc@refs/convert/parquet/"
               "multiple_choice/validation/0000.parquet")

def eval_truthfulqa(model, tok, n=N_TRUTHFULQA):
    ds = load_dataset("parquet", data_files=TQA_PARQUET, split="train")
    ds = ds.select(range(min(n, len(ds))))
    correct = []
    for ex in ds:
        q, choices, label = ex["question"], ex["choices"], int(ex["label"])
        scores = np.array([seq_logprob(model, tok, q, c) for c in choices])
        correct.append(float(int(scores.argmax()) == label))
    return float(np.mean(correct))

rows_tqa = []
for short, hf_id in MODEL_MAP.items():
    print("TruthfulQA-MC ==>", short)
    tok, model = load_model(hf_id)
    acc = eval_truthfulqa(model, tok)
    rows_tqa.append({"Model": short, "TQA_acc": acc})
    del model; torch.cuda.empty_cache()
tqa_df = pd.DataFrame(rows_tqa); tqa_df.to_csv("ext_truthfulqa.csv", index=False)
tqa_df.round(3)

In [ ]:
import re
from collections import Counter
from datasets import load_dataset

def _norm(s): return re.sub(r"\s+", " ", str(s).strip())
def exact_match(pred, gold): return float(_norm(pred).lower() == _norm(gold).lower())
def token_f1(pred, gold):
    p, g = _norm(pred).lower().split(), _norm(gold).lower().split()
    if not p or not g: return float(p == g)
    ov = sum((Counter(p) & Counter(g)).values())
    if ov == 0: return 0.0
    prec, rec = ov/len(p), ov/len(g)
    return 2*prec*rec/(prec+rec)

@torch.no_grad()
def gen_text(model, tok, prompt, max_new_tokens=32):
    ids = tok(prompt, return_tensors="pt", truncation=True, max_length=1024).to(model.device)
    out = model.generate(**ids, max_new_tokens=max_new_tokens, do_sample=False,
                         pad_token_id=tok.pad_token_id)
    return tok.decode(out[0, ids["input_ids"].shape[1]:], skip_special_tokens=True).strip()

def first_line(s): return s.split("\n")[0].strip()

def eval_dyck(model, tok, n=N_SYMB, k=K_SHOT):
    ds = load_dataset("lukaemon/bbh", "dyck_languages", split="test")
    demos = ds.select(range(k))
    test  = ds.select(range(k, min(k+n, len(ds))))
    shots = "\n\n".join(f"{d['input']}\n{d['target']}" for d in demos)
    ems, f1s = [], []
    for ex in test:
        pred = first_line(gen_text(model, tok, f"{shots}\n\n{ex['input']}\n", max_new_tokens=24))
        ems.append(exact_match(pred, ex["target"])); f1s.append(token_f1(pred, ex["target"]))
    return float(np.mean(ems)), float(np.mean(f1s))

def eval_listfn(model, tok, n=N_SYMB):
    # 'inputs' already embeds the induction examples (self-contained few-shot).
    ds = load_dataset("tasksource/bigbench", "list_functions", split="validation").select(range(n))
    ems = [exact_match(first_line(gen_text(model, tok, ex["inputs"], max_new_tokens=48)),
                        ex["targets"][0]) for ex in ds]
    return float(np.mean(ems))

rows_symb = []
for short, hf_id in MODEL_MAP.items():
    print("Symbolic ==>", short)
    tok, model = load_model(hf_id)
    dyck_em, dyck_f1 = eval_dyck(model, tok)
    lf_em = eval_listfn(model, tok)
    rows_symb.append({"Model": short, "dyck_em": dyck_em, "dyck_f1": dyck_f1, "listfn_em": lf_em})
    print(f"   dyck_f1={dyck_f1:.3f} dyck_em={dyck_em:.3f} listfn_em={lf_em:.3f}")
    del model; torch.cuda.empty_cache()
symb_df = pd.DataFrame(rows_symb); symb_df.to_csv("ext_symbolic.csv", index=False)
symb_df.round(3)

In [ ]:
from functools import reduce
frames = [lpp[["Model","EntropyFloor","PR","ER","SPC","AR","IFEval","BBH","MMLU-PRO"]]]
# ext_symbolic.csv (this notebook) + optional published scores; NB3's normalized_metrics gives the
# winsorized PR/ER so we can check the symbolic links survive outlier removal too.
for f in ["external_scores.csv","ext_truthfulqa.csv","ext_symbolic.csv","normalized_metrics.csv"]:
    if os.path.exists(f):
        d = pd.read_csv(f).dropna(axis=1, how="all")
        keep = [c for c in d.columns if c == "Model" or c not in
                set().union(*[set(fr.columns) for fr in frames])]
        if len(keep) > 1: frames.append(d[keep])
data = reduce(lambda a,b: a.merge(b, on="Model", how="left"), frames)
data.to_csv("nb1_merged.csv", index=False)
print("Merged columns:", list(data.columns))
data.round(3)

In [ ]:
from scipy.stats import spearmanr

def boot_spearman(x, y, n_boot=5000):
    x, y = np.asarray(x, float), np.asarray(y, float)
    m = np.isfinite(x) & np.isfinite(y); x, y = x[m], y[m]
    if len(x) < 4: return (np.nan, np.nan, np.nan, np.nan, len(x))
    rho, p = spearmanr(x, y)
    idx = np.arange(len(x)); boots = []
    for _ in range(n_boot):
        b = np.random.choice(idx, len(idx), replace=True)
        if len(set(x[b])) > 1 and len(set(y[b])) > 1:
            boots.append(spearmanr(x[b], y[b])[0])
    lo, hi = np.nanpercentile(boots, [2.5, 97.5]) if boots else (np.nan, np.nan)
    return rho, p, lo, hi, len(x)

# hypothesis map: LPP metric -> external targets it should predict (expect NEGATIVE for PR/ER)
hyp = {
    "EntropyFloor": ["TQA_acc","TruthfulQA_MC2","HaluEval_acc"],   # calibration -> hallucination
    "PR":           ["dyck_f1","dyck_em","listfn_em"],            # compactness -> symbolic generalization
    "ER":           ["dyck_f1","dyck_em","listfn_em"],
    "PR_robust":    ["dyck_f1","listfn_em"],                      # outlier-winsorized (survives?)
    "ER_robust":    ["dyck_f1","listfn_em"],
}
res = []
for lm, targets in hyp.items():
    for t in targets:
        if t in data.columns:
            rho,p,lo,hi,n = boot_spearman(data[lm], data[t])
            res.append({"LPP_metric":lm,"external_target":t,"n":n,
                        "spearman_rho":round(rho,3),"p":round(p,3),
                        "CI95_low":round(lo,3),"CI95_high":round(hi,3)})
res = pd.DataFrame(res)
res.to_csv("nb1_correlations.csv", index=False)
print("NOTE: n=8 models -> low statistical power; interpret via effect size + CI, not p alone.\n")
res

In [ ]:
# Scatter plots for the strongest hypothesized links
import matplotlib.pyplot as plt
pairs = [("EntropyFloor","TQA_acc","less hallucination"),
         ("ER","dyck_f1","better structural completion"),
         ("PR","listfn_em","better rule induction")]
pairs = [(a,b,c) for a,b,c in pairs if b in data.columns]
if pairs:
    fig, axes = plt.subplots(1, len(pairs), figsize=(5*len(pairs),4.3))
    if len(pairs)==1: axes=[axes]
    for ax,(a,b,note) in zip(axes,pairs):
        d = data[["Model",a,b]].dropna()
        ax.scatter(d[a], d[b])
        for _,r in d.iterrows(): ax.annotate(r["Model"], (r[a],r[b]), fontsize=8)
        rho,_ = spearmanr(d[a], d[b]) if len(d)>3 else (np.nan,np.nan)
        ax.set_xlabel(a); ax.set_ylabel(b); ax.set_title(f"{a} vs {b}\n(rho={rho:.2f}; higher {b}={note})")
    plt.tight_layout(); plt.savefig("nb1_external_scatter.png", dpi=200); plt.show()
else:
    print("No computed/published external targets yet — run Path A or Path B first.")

In [ ]:
from scipy.stats import rankdata, pearsonr
SIZE_B = {"Qwen-0.5B":0.5,"Qwen-1.5B":1.5,"Qwen-3B":3,"Qwen-7B":7,"Qwen-14B":14,
          "Llama-3B":3,"Llama-8B":8,"Mistral-7B":7}
data["size_b"] = data["Model"].map(SIZE_B)

def partial_spearman(a, b, ctrl="size_b"):
    d = data[[a, b, ctrl]].dropna()
    if len(d) < 5: return np.nan, np.nan, len(d)
    R = {c: rankdata(d[c].values) for c in (a, b, ctrl)}
    Z = np.column_stack([np.ones(len(d)), R[ctrl]])
    resid = lambda v: v - Z @ np.linalg.lstsq(Z, v, rcond=None)[0]
    r, p = pearsonr(resid(R[a]), resid(R[b]))
    return round(float(r), 3), round(float(p), 3), len(d)

checks = [("EntropyFloor","TQA_acc"), ("ER","dyck_f1"), ("PR","dyck_f1"),
          ("ER","listfn_em"), ("PR","listfn_em"), ("ER","SPC"), ("PR","SPC")]
prows = []
for a, b in checks:
    if a in data.columns and b in data.columns:
        rho0, p0, _ = boot_spearman(data[a], data[b])[:2] + (0,)
        rp, pp, n = partial_spearman(a, b)
        prows.append({"pair": f"{a} ~ {b}", "spearman": round(rho0,3),
                      "partial_ctrl_size": rp, "partial_p": pp, "n": n})
pcorr = pd.DataFrame(prows); pcorr.to_csv("nb1_partial_correlations.csv", index=False)
print("If 'partial_ctrl_size' keeps the same sign and stays sizeable, the LPP link is not just scale.\n")
pcorr